In [1]:
import os

WORK_DIR = "/content"
CACHE_DIR = "/content/hf"

os.makedirs(CACHE_DIR, exist_ok=True)
os.environ["HF_HOME"] = f"{CACHE_DIR}"

In [2]:
!wget https://ftp.ncbi.nlm.nih.gov/pub/datasets/command-line/LATEST/linux-amd64/datasets
!chmod +x datasets
!sudo mv datasets /usr/local/bin/

--2026-05-30 15:41:21--  https://ftp.ncbi.nlm.nih.gov/pub/datasets/command-line/LATEST/linux-amd64/datasets
Resolving ftp.ncbi.nlm.nih.gov (ftp.ncbi.nlm.nih.gov)... 130.14.250.11, 130.14.250.12, 2607:f220:41e:250::7, ...
Connecting to ftp.ncbi.nlm.nih.gov (ftp.ncbi.nlm.nih.gov)|130.14.250.11|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 19673603 (19M)
Saving to: ‘datasets’

datasets            100%[===================>]  18.76M  56.4MB/s    in 0.3s    

2026-05-30 15:41:21 (56.4 MB/s) - ‘datasets’ saved [19673603/19673603]



In [3]:
!datasets version

datasets version: 18.29.1


In [4]:
!pip install ncbi-datasets-pylib

In [5]:
import pandas as pd

gcs_csv = "gs://rojo_project/isolation_source_labels.csv"

df_ncbi_filter = pd.read_csv(gcs_csv)

print(df_ncbi_filter.shape)

assemblies = (
    df_ncbi_filter["Assembly"]
    .dropna()
    .astype(str)
    .unique()
)

print("Unique assemblies:", len(assemblies))

(32324, 2)
Unique assemblies: 32324


## Fasta file

In [6]:
# fasta files
import subprocess
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed


# bucket OUTPUT PATH

GCS_BUCKET = "gs://rojo_project/vfdb/fasta_file"

# Local fast temp storage
temp_root = Path("/content/fasta_temp")
temp_root.mkdir(parents=True, exist_ok=True)

assemblies = df_ncbi_filter["Assembly"].dropna().astype(str).unique()


# WORKER FUNCTION

def download_and_upload(acc):
    try:
        work_dir = temp_root / acc
        work_dir.mkdir(parents=True, exist_ok=True)

        zip_path = work_dir / "ncbi_dataset.zip"

        # download genome
        subprocess.run([
            "datasets", "download", "genome", "accession", acc,
            "--include", "genome",
            "--filename", str(zip_path)
        ], check=True, capture_output=True)

        # unzip locally
        subprocess.run([
            "unzip", "-q", str(zip_path), "-d", str(work_dir)
        ], check=True)

        # find fasta
        fasta = next(work_dir.rglob("*.fna"), None)
        if not fasta:
            return f"NO FASTA: {acc}"

        # upload to bucket directly
        gcs_path = f"{GCS_BUCKET}/{acc}.fna"

        subprocess.run([
            "gsutil", "cp", str(fasta), gcs_path
        ], check=True, capture_output=True)

        return f"UPLOADED: {acc}"

    except Exception as e:
        return f"FAILED: {acc} -> {str(e)[:80]}"



# PARALLEL EXECUTION

print(f"Processing {len(assemblies)} genomes → {GCS_BUCKET}")

results = []

with ThreadPoolExecutor(max_workers=8) as executor:
    futures = [executor.submit(download_and_upload, acc) for acc in assemblies]

    for i, f in enumerate(as_completed(futures), 1):
        msg = f.result()
        results.append(msg)

        if i % 20 == 0:
            print(f"[{i}/{len(assemblies)}] done")

print("\nDONE")
print("Total genomes:", len(assemblies))

Processing 32324 genomes → gs://rojo_project/vfdb/fasta_file
[20/32324] done
[40/32324] done
[60/32324] done
[80/32324] done
[100/32324] done
[120/32324] done
[140/32324] done
[160/32324] done
[180/32324] done
[200/32324] done
[220/32324] done
[240/32324] done
[260/32324] done
[280/32324] done
[300/32324] done
[320/32324] done
[340/32324] done
[360/32324] done
[380/32324] done
[400/32324] done
[420/32324] done
[440/32324] done
[460/32324] done
[480/32324] done
[500/32324] done
[520/32324] done
[540/32324] done
[560/32324] done
[580/32324] done
[600/32324] done
[620/32324] done
[640/32324] done
[660/32324] done
[680/32324] done
[700/32324] done
[720/32324] done
[740/32324] done
[760/32324] done
[780/32324] done
[800/32324] done
[820/32324] done
[840/32324] done
[860/32324] done
[880/32324] done
[900/32324] done
[920/32324] done
[940/32324] done
[960/32324] done
[980/32324] done
[1000/32324] done
[1020/32324] done
[1040/32324] done
[1060/32324] done
[1080/32324] done
[1100/32324] done
[1

## BLAST

In [7]:

# CONFIG


import os, subprocess, gzip, shutil, csv
from pathlib import Path


GCS_BUCKET = "gs://rojo_project/vfdb/fasta_file"

WORK_DIR      = Path("/content/vfdb_pipeline")
GENOME_DIR    = Path("/content/genomes")
VFDB_DIR      = WORK_DIR / "vfdb"
BLAST_DIR     = WORK_DIR / "blast_results1"
CONTEXT_DIR   = WORK_DIR / "context_fastas1"
RESULTS_DIR   = Path("/content/vfdb_context")

CONTEXT_WINDOW = 1000
MIN_IDENTITY   = 80.0
MIN_COVERAGE   = 60.0
EVALUE_THRESH  = "1e-5"

for d in [VFDB_DIR, BLAST_DIR, CONTEXT_DIR, RESULTS_DIR, GENOME_DIR]:
    d.mkdir(parents=True, exist_ok=True)


In [8]:

# INSTALL DEPENDENCIES

print("Installing BLAST + Biopython...")

subprocess.run(["apt-get", "install", "-y", "-q", "ncbi-blast+"], check=True)
subprocess.run(["pip", "install", "-q", "biopython"], check=True)

print("Dependencies ready ")

Installing BLAST + Biopython...
Dependencies ready 


In [9]:

#  SYNC GENOMES FROM GCS

print("Syncing genomes (deduplicated)...")

subprocess.run([
    "gsutil",
    "-m",
    "rsync",
    "-r",
    GCS_BUCKET,
    str(GENOME_DIR)
], check=True)

genomes = sorted(GENOME_DIR.glob("*.fna"))
print(f"Genomes loaded: {len(genomes)}")

Syncing genomes (deduplicated)...
Genomes loaded: 32191


In [10]:

#  DOWNLOAD + BUILD VFDB DATABASE


VFDB_GZ    = VFDB_DIR / "VFDB_setB_nt.fas.gz"
VFDB_FASTA = VFDB_DIR / "VFDB_setB_nt.fas"
VFDB_DB    = VFDB_DIR / "vfdb"

if not VFDB_FASTA.exists():
    print("Downloading VFDB...")
    subprocess.run([
        "wget", "-q", "-O", str(VFDB_GZ),
        "http://www.mgc.ac.cn/VFs/Down/VFDB_setB_nt.fas.gz"
    ], check=True)

    with gzip.open(VFDB_GZ, "rb") as f_in, open(VFDB_FASTA, "wb") as f_out:
        shutil.copyfileobj(f_in, f_out)

print("Building BLAST DB...")
subprocess.run([
    "makeblastdb",
    "-in", str(VFDB_FASTA),
    "-dbtype", "nucl",
    "-out", str(VFDB_DB),
    "-title", "VFDB_setB"
], check=True)

print("VFDB ready ")

Building BLAST DB...
VFDB ready 


In [12]:

# PARALLEL BLAST SEARCH


from concurrent.futures import ThreadPoolExecutor, as_completed
import subprocess
from pathlib import Path

BLAST_FMT = (
    "6 qseqid qstart qend qlen sseqid sstart send slen "
    "pident length mismatch gapopen evalue bitscore stitle"
)


# PERFORMANCE SETTINGS


MAX_WORKERS   = 16   # number of parallel genomes
BLAST_THREADS = 2    # per blastn job

GCS_BLAST_BUCKET = "gs://rojo_project/vfdb/vfdb_context"

failed = []
skipped = 0
completed = 0


print(f"\nRunning PARALLEL BLAST on {len(genomes)} genomes...")
print(f"Workers={MAX_WORKERS}, Threads/job={BLAST_THREADS}\n")



# BLAST FUNCTION


def run_blast(genome_path):

    out_tsv = BLAST_DIR / f"{genome_path.stem}.blast.tsv"

    # Skip if already done locally OR in bucket
    if out_tsv.exists() and out_tsv.stat().st_size > 0:
        return ("skipped", genome_path.name)

    cmd = [
        "blastn",
        "-query", str(genome_path),
        "-db", str(VFDB_DB),
        "-out", str(out_tsv),
        "-outfmt", BLAST_FMT,
        "-evalue", EVALUE_THRESH,
        "-perc_identity", str(MIN_IDENTITY),
        "-num_threads", str(BLAST_THREADS),
        "-max_target_seqs", "10"
    ]

    try:
        subprocess.run(cmd, check=True,
                       stdout=subprocess.DEVNULL,
                       stderr=subprocess.DEVNULL)

       # save
        subprocess.run([
            "gsutil", "-q", "cp",
            str(out_tsv),
            GCS_BLAST_BUCKET + "/"
        ])

        return ("done", genome_path.name)

    except subprocess.CalledProcessError:
        return ("failed", genome_path.name)


# PARALLEL EXECUTION


with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:

    futures = [executor.submit(run_blast, g) for g in genomes]

    for future in as_completed(futures):

        status, name = future.result()
        completed += 1

        if status == "failed":
            failed.append(name)

        elif status == "skipped":
            skipped += 1

        if completed % 25 == 0:
            print(f"[{completed}/{len(genomes)}] processed")

print("\nBLAST COMPLETE ✓")
print(f"Skipped : {skipped}")
print(f"Failed  : {len(failed)}")


Running PARALLEL BLAST on 32191 genomes...
Workers=16, Threads/job=2

[25/32191] processed
[50/32191] processed
[75/32191] processed
[100/32191] processed
[125/32191] processed
[150/32191] processed
[175/32191] processed
[200/32191] processed
[225/32191] processed
[250/32191] processed
[275/32191] processed
[300/32191] processed
[325/32191] processed
[350/32191] processed
[375/32191] processed
[400/32191] processed
[425/32191] processed
[450/32191] processed
[475/32191] processed
[500/32191] processed
[525/32191] processed
[550/32191] processed
[575/32191] processed
[600/32191] processed
[625/32191] processed
[650/32191] processed
[675/32191] processed
[700/32191] processed
[725/32191] processed
[750/32191] processed
[775/32191] processed
[800/32191] processed
[825/32191] processed
[850/32191] processed
[875/32191] processed
[900/32191] processed
[925/32191] processed
[950/32191] processed
[975/32191] processed
[1000/32191] processed
[1025/32191] processed
[1050/32191] processed
[1075

In [13]:

# 5. PARSE BLAST + CONTEXT FUNCTIONS


BLAST_COLS = [
    "qseqid","qstart","qend","qlen","sseqid","sstart","send","slen",
    "pident","length","mismatch","gapopen","evalue","bitscore","stitle"
]

def parse_blast(tsv_path, min_cov=MIN_COVERAGE):
    hits = []
    with open(tsv_path) as fh:
        for line in fh:
            parts = line.strip().split("\t")
            if len(parts) < len(BLAST_COLS):
                continue

            d = dict(zip(BLAST_COLS, parts))

            qlen = int(d["qlen"])
            alen = int(d["length"])
            qcov = (alen / qlen * 100) if qlen > 0 else 0

            if qcov < min_cov:
                continue

            d["qcov"] = round(qcov, 2)
            d["hit_start"] = min(int(d["qstart"]), int(d["qend"]))
            d["hit_end"]   = max(int(d["qstart"]), int(d["qend"]))

            hits.append(d)

    return hits


def extract_context(seq, start, end, window, contig_len):
    ctx_start = max(1, start - window)
    ctx_end   = min(contig_len, end + window)
    return ctx_start, ctx_end, str(seq[ctx_start-1:ctx_end])


In [14]:

# CONTEXT EXTRACTION

from Bio import SeqIO
from Bio.SeqRecord import SeqRecord
from Bio.Seq import Seq

summary_rows = []
genomes_with_hits = 0
total_contexts = 0

print("\nExtracting contexts...\n")

for i, genome_path in enumerate(genomes, 1):

    blast_file = BLAST_DIR / f"{genome_path.stem}.blast.tsv"
    if not blast_file.exists() or blast_file.stat().st_size == 0:
        continue

    hits = parse_blast(blast_file)
    if not hits:
        continue

    genome_seqs = SeqIO.to_dict(SeqIO.parse(str(genome_path), "fasta"))

    seen = set()
    context_records = []

    for hit in hits:

        contig = hit["qseqid"]
        if contig not in genome_seqs:
            continue

        seq_record = genome_seqs[contig]
        contig_len = len(seq_record.seq)

        ctx_start, ctx_end, subseq = extract_context(
            seq_record.seq,
            hit["hit_start"],
            hit["hit_end"],
            CONTEXT_WINDOW,
            contig_len
        )

        key = (contig, ctx_start, ctx_end)
        if key in seen:
            continue
        seen.add(key)

        record_id = (
            f"{genome_path.stem}|{contig}|ctx:{ctx_start}-{ctx_end}"
            f"|VF:{hit['sseqid']}|id:{hit['pident']}%"
            f"|cov:{hit['qcov']}%|hit:{hit['hit_start']}-{hit['hit_end']}"
        )

        context_records.append(
            SeqRecord(Seq(subseq), id=record_id, description=hit["stitle"][:100])
        )

        summary_rows.append(hit)

    if context_records:
        out_fasta = CONTEXT_DIR / f"{genome_path.stem}.fasta"
        SeqIO.write(context_records, str(out_fasta), "fasta")

        genomes_with_hits += 1
        total_contexts += len(context_records)

    if i % 50 == 0:
        print(f"[{i}/{len(genomes)}] processed")

print("\nContext extraction done")



Extracting contexts...

[50/32191] processed
[100/32191] processed
[150/32191] processed
[250/32191] processed
[300/32191] processed
[350/32191] processed
[400/32191] processed
[450/32191] processed
[500/32191] processed
[550/32191] processed
[600/32191] processed
[650/32191] processed
[700/32191] processed
[750/32191] processed
[800/32191] processed
[850/32191] processed
[900/32191] processed
[950/32191] processed
[1050/32191] processed
[1100/32191] processed
[1150/32191] processed
[1200/32191] processed
[1250/32191] processed
[1300/32191] processed
[1350/32191] processed
[1450/32191] processed
[1500/32191] processed
[1550/32191] processed
[1700/32191] processed
[1750/32191] processed
[1800/32191] processed
[1850/32191] processed
[1950/32191] processed
[2000/32191] processed
[2050/32191] processed
[2100/32191] processed
[2200/32191] processed
[2250/32191] processed
[2300/32191] processed
[2350/32191] processed
[2500/32191] processed
[2550/32191] processed
[2650/32191] processed
[2700

In [15]:

# SAVE SUMMARY


summary_tsv = WORK_DIR / "vfdb_hits_summary.tsv"

if summary_rows:
    with open(summary_tsv, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=summary_rows[0].keys(), delimiter="\t")
        writer.writeheader()
        writer.writerows(summary_rows)



# 8. SAVE RESULTS TO BUCKET


RESULTS_BUCKET = "gs://rojo_project/vfdb/vfdb_context"

print("\nUploading results to GCS...")

# Upload context FASTA files
context_fastas = list(CONTEXT_DIR.glob("*.fasta"))

if context_fastas:
    subprocess.run([
        "gsutil",
        "-m",
        "cp",
        str(CONTEXT_DIR / "*.fasta"),
        RESULTS_BUCKET
    ], check=True)

    print(f"Uploaded {len(context_fastas)} FASTA files")

# Upload summary TSV
if summary_tsv.exists():
    subprocess.run([
        "gsutil",
        "cp",
        str(summary_tsv),
        RESULTS_BUCKET
    ], check=True)

    print("Uploaded summary TSV ")


Uploading results to GCS...
Uploaded 29920 FASTA files
Uploaded summary TSV 


In [16]:
print(f"""
╔══════════════════════════════════════════════════╗
║              Pipeline Complete ✓                 ║
╠══════════════════════════════════════════════════╣
║  Genomes scanned    : {len(genomes):<5}                        ║
║  Genomes with hits  : {genomes_with_hits:<5}                        ║
║  Context sequences  : {total_contexts:<5}                        ║
║  Window size        : ±{CONTEXT_WINDOW} bp                    ║
║                                                  ║
║  Results uploaded to:                           ║
║  gs://rojo_project/vfdb/vfdb_context                ║
╚══════════════════════════════════════════════════╝
""")


╔══════════════════════════════════════════════════╗
║              Pipeline Complete ✓                 ║
╠══════════════════════════════════════════════════╣
║  Genomes scanned    : 32191                        ║
║  Genomes with hits  : 29920                        ║
║  Context sequences  : 283769                        ║
║  Window size        : ±1000 bp                    ║
║                                                  ║
║  Results uploaded to:                           ║
║  gs://rojo_project/vfdb/vfdb_context                ║
╚══════════════════════════════════════════════════╝

